In [1]:
import sys
from pathlib import Path
workspace_root = Path().cwd().parent
print(f"Workspace root: {workspace_root}")
sys.path.insert(0, str(workspace_root))

Workspace root: e:\Project\news_secretary\apps\back


In [2]:
import os

from app.packages.collect_news import collect_article_pipeline

In [3]:
from app.configuration import BRAVE_SEARCH_API_KEY

print(f"BRAVE_SEARCH_API_KEY: {BRAVE_SEARCH_API_KEY[:10]}")

BRAVE_SEARCH_API_KEY: BSAyfCv_c0


In [4]:


query_list = ["petaling jaya selangor"]

collect_article_pipeline(
    query_list=query_list,
    time_range="pd",
    search_limit=3
    )

2026-10-29 22:27:45,128 - app.packages.collect_news - ERROR - Error inserting article Agencies test emergency response plans | The Star: Key (link)=(https://www.thestar.com.my/metro/metro-news/2026/10/29/agencies-test-emergency-response-plans) already exists.
2026-10-29 22:27:45,159 - app.packages.collect_news - INFO - Existing record: {'id': UUID('01a187da-0d6a-7dcb-9654-c1be6e85af1b'), 'title': 'Agencies test emergency response plans | The Star', 'link': 'https://www.thestar.com.my/metro/metro-news/2026/10/29/agencies-test-emergency-response-plans', 'snippet': 'Security agencies in Selangor convened in Petaling Jaya for a joint tabletop exercise to test emergency response plans during a simulated public order threat.\nAhmad Fadzli (left) pointing at a map used in the tabletop exercise simulating a riot in Petaling Jaya. — ART CHEN/The Star · Security agencies in Selangor convened in Petaling Jaya for a joint tabletop exercise to test emergency response plans during a simulated public

{UUID('01a187da-0d6a-7dcb-9654-c1be6e85af1b'): ArticleModel(id=UUID('01a187da-0d6a-7dcb-9654-c1be6e85af1b'), title='Agencies test emergency response plans | The Star', link='https://www.thestar.com.my/metro/metro-news/2026/10/29/agencies-test-emergency-response-plans', snippet='Security agencies in Selangor convened in Petaling Jaya for a joint tabletop exercise to test emergency response plans during a simulated public order threat.\nAhmad Fadzli (left) pointing at a map used in the tabletop exercise simulating a riot in Petaling Jaya. — ART CHEN/The Star · Security agencies in Selangor convened in Petaling Jaya for a joint tabletop exercise to test emergency response plans during a simulated public order threat.\nSelangor State Secretary Datuk Dr Ahmad Fadzli Ahmad Tajuddin said such exercises allowed agencies to identify weaknesses, gaps, overlapping roles or communication problems before a real incident occurs. “The findings from this exercise must be translated into follow-up acti

In [4]:
from app.db.connect_db import connect_to_postgres
conn2 = connect_to_postgres()   # 全新连接，不复用
cur = conn2.cursor()
cur.execute("SELECT 1")
print(cur.fetchone())
conn2.rollback()   # 清理
conn2.close()


(1,)


In [ ]:
import sqlite3
from app.configuration import SQLITE_DB_PATH, TABLE_NAMES


def retrieve_leaves_keywords_from_db():
    with sqlite3.connect(SQLITE_DB_PATH) as conn:
        cursor = conn.cursor(row_factory=dict_row)
        cursor.execute(f"SELECT * FROM {TABLE_NAMES['keywords']} WHERE parent_id IS NULL")
        keywords = cursor.fetchall()
    return keywords

def structure_keywords(keywords):
    keyword_dict = {}
    for keyword in keywords:
        # ('medtop:20000893', 'artistic cycling', 'Individual or teams on one or two bicycles in an arena. Points are awarded by a jury.', 'arena cycling , bicycle competition , team cycling', 'medtop:20000892', None, '2026-10-16 06:40:33')
        keyword_id, name, description, synonyms, parent_id, child_ids, created_at = keyword
        keyword_dict[keyword_id] = {
            "keyword": name,
            "description": description,
            "synonyms": synonyms.split(" , ") if synonyms else [],
            "parent_id": parent_id,
            "child_ids": child_ids.split(" , ") if child_ids else [],
            "created_at": created_at
        }
    return keyword_dict

def keywords_structure_to_text(keyword_dict):
    keywords_texts = []
    for _, details in keyword_dict.items():
        keyword_name = details["keyword"]
        keyword_description = details["description"]
        keyword_synonyms = details["synonyms"]
        parents = []
        parent_id = details["parent_id"]
        while parent_id:
            parent_details = keyword_dict.get(parent_id)
            if parent_details:
                parents.append(parent_details["keyword"])
                parent_id = parent_details["parent_id"]
            else:
                break
        
        text = f"Keyword: {keyword_name}\nDescription: {keyword_description}\nSynonyms: {', '.join(keyword_synonyms)}\nParents: {' > '.join(parents[::-1])}\n"
        keywords_texts.append(text)
    return keywords_texts

keywords = retrieve_leaves_keywords_from_db()
keyword_dict = structure_keywords(keywords)
keywords_texts = keywords_structure_to_text(keyword_dict)
keywords_texts[:5]  # Display the first 5 keywords for verification

In [ ]:
from scipy.